# 🟡 Lesson 13 — Earth Engine API

**Level: Intermediate** · Google's planetary-scale catalogue — query decades of satellite data without downloading a single scene.

*Part of the companion package for [python_for_geologists](https://github.com/kevinalexandr19/python_for_geologists) by Kevin Alexander Gomez.*

> ⚠️ **This notebook is not executed in this repo** — it requires a (free) Google Earth Engine account and one-time authentication. The code is complete and ready to run once you have set it up.

**Setup (once):**
```bash
pip install earthengine-api
```
Then register at https://code.earthengine.google.com/register and run the first cell.

In [ ]:
import ee

ee.Authenticate()                 # opens a browser the first time
ee.Initialize(project="your-project-id")
print(ee.String("Earth Engine ready").getInfo())

## 1. Elevation at a point — SRTM without downloading it

In [ ]:
srtm = ee.Image("USGS/SRTMGL1_003")
misti = ee.Geometry.Point([-71.409, -16.294])

elev = srtm.sample(misti, scale=30).first().get("elevation")
print("Misti summit area elevation:", elev.getInfo(), "m")

## 2. Regional statistics — mean elevation of a study area

In [ ]:
aoi = ee.Geometry.Rectangle([-72.0, -16.8, -71.0, -16.0])

stats = srtm.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=aoi, scale=90, maxPixels=1e9)
print(stats.getInfo())

## 3. Cloud-free Sentinel-2 composite + NDVI

In [ ]:
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate("2024-01-01", "2024-12-31")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10)))

print("scenes found:", s2.size().getInfo())

composite = s2.median()
ndvi = composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
mean_ndvi = ndvi.reduceRegion(ee.Reducer.mean(), aoi, scale=100).get("NDVI")
print("mean NDVI 2024:", mean_ndvi.getInfo())

## 4. Export a clipped DEM to Google Drive

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=srtm.clip(aoi), description="srtm_arequipa",
    folder="earthengine", region=aoi, scale=30, maxPixels=1e9)
task.start()
print("export started - check https://code.earthengine.google.com/tasks")

### ✏️ Try it
1. Swap SRTM for `NASA/NASADEM_HGT/001` and compare the point elevation.
2. Compute a **NDWI** (`B3`,`B8`) composite for the wet vs dry season and difference them.

📚 Docs: https://developers.google.com/earth-engine/guides/python_install